In [ ]:
import requests
import pandas as pd
import os
import re
from tqdm import tqdm #barra de progreso
from dotenv import load_dotenv 

pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None) #para ver toda la tabla, en nuestro caso son solo 30 se supone que no tenemos que tener problema

API_KEY = '2c6d07b262d82febbd27cf5327a36d55' 

artistas = [
    "La Fuga",
    "Héroes del Silencio",
    "Billie Eilish",
    "Love of Lesbian",
    "Estopa",
    "Mägo de Oz",
    "Mr. Kilombo",
    "Rozalén",
    "Taburete",
    "Extremoduro",
    "La Plazuela",
    "Veintiuno",
    "Ojete Calor",
    "Rata Blanca",
    "Vetusta Morla",
    "Leiva",
    "Bad Bunny",
    "Enrique Bunbury",
    "Marea",
    "Joaquín Sabina",
    "Rosalía",
    "Queen",
    "The Lumineers",
    "Foo Fighters",
    "Muse",
    "Metallica",
    "Ginebras",
    "IZAL",
    "Kaiser Chiefs",
    "Residente"
]

artistas = [a.strip() for a in artistas]

#Funcion para llamar a la API
def obtener_info_artista(nombre):
    url = "http://ws.audioscrobbler.com/2.0/"
    params = {
        "method": "artist.getinfo",
        "artist": nombre,
        "api_key": API_KEY,
        "format": "json",
        "autocorrect" : 1,  
        "lang": "es"
     }
    
    response = requests.get(url, params=params, timeout=10)
    response.raise_for_status() #lanza excepción si hay error HTTP
    return response.json()

def procesar_artista(nombre):
    try:
        data = obtener_info_artista(nombre)
        artista = data["artist"]
        
        bio = artista["bio"]["summary"]
        listeners = int(artista["stats"]["listeners"])
        playcount = int(artista["stats"]["playcount"])
        similares = [a["name"] for a in artista["similar"]["artist"]]
        
        return {
            "artista": nombre,
            "biografia": bio,
            "listeners": listeners,
            "playcount": playcount,
            "similares": ", ".join(similares)
        }
    
    except KeyError as e:
        print(f"[KeyError] '{nombre}': clave no encontrada {e}")  #except especifico que muestra qué clave faltó
        return None
    except requests.RequestException as e:
        print(f"[RequestError] '{nombre}': {e}")
        return None

In [37]:
resultados = []
bar = tqdm(artistas, desc="Iniciando...")
 
for artista in bar:
    bar.set_description(f"Procesando: {artista}")
    info = procesar_artista(artista)
    if info:
        resultados.append(info)
 
print(f"\n✅ {len(resultados)}/{len(artistas)} artistas obtenidos correctamente")

Procesando: Residente: 100%|██████████| 30/30 [00:05<00:00,  5.29it/s]         


✅ 30/30 artistas obtenidos correctamente


In [ ]:
#Crear DataFrame
df_lastfm = pd.DataFrame(resultados)
df_lastfm.head() #con esto veo 5

In [ ]:
df_lastfm #con esto sale toda la tabla

In [ ]:
#para verlos en orden de más popular a menos 
df_lastfm.sort_values(by="listeners", ascending=False) 

In [29]:
# ── MEJORA: exportar a CSV para no tener que volver a llamar a la API ──
df_lastfm.to_csv("lastfm_artistas.csv", index=False, encoding="utf-8-sig")
print("✅ Datos guardados en lastfm_artistas.csv")

✅ Datos guardados en lastfm_artistas.csv


In [ ]:
#merge con Deezer (cuando tengas df_deezer disponible)
df_final = df_deezer.merge(df_lastfm, on="artista", how="left")
df_final.head() 